# Benchmark embedders para semantic similarity search

In [4]:
import os
import sys
import numpy as np
import pandas as pd


from langchain_community.embeddings import LlamaCppEmbeddings



from langchain_core.messages import HumanMessage

In [2]:
sys.path.append('../src/')
%load_ext autoreload
%autoreload 2

from utils import *
from llm_factory import *

# Procesando documentos

In [5]:

#Inicializando bbdd
PERSIST_DIRECTORY = "../db_chroma"
MODEL_CACHE_DIR = "../model_cache"
SOURCE_DOCS_PATH = "../data/raw" 

all_docs = [] # <--- Lista para acumular todos los documentos

for filename in os.listdir(SOURCE_DOCS_PATH): # <--- Iteramos sobre los archivos
    if filename.endswith(".md"): # <--- Filtramos solo archivos .md
        file_path = os.path.join(SOURCE_DOCS_PATH, filename) # <--- Construimos la ruta completa
        processed_document_chunks = process_markdown_document(file_path) # <--- Procesamos cada archivo
        all_docs.extend(processed_document_chunks) # <--- Agregamos los documentos
        
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=100,
    separators=["\n\n"])

chunked_splits = text_splitter.split_documents(all_docs)
print(f"Generated {len(chunked_splits)} chunks")
# Add metadata
for i, doc in enumerate(chunked_splits):
    doc.metadata["doc_id"] = f"chunk_{i}"

pd.DataFrame(chunked_splits).to_csv("./chunks.csv")


Generated 28 chunks


# Generacion de dataset de pruebas

El dataset contiene pares: pregunta, ID del chunk con la respuesta.

En la prueba se envia al vectorstore la pregunta y se evalua si el chunk ID con la respuesta esta contenido en la salida del vectorstore.

En esta fase se genera una pregunta para cada chunk usando GenAI (puede alucinar).

In [6]:
try:
    my_llm_key = get_api_key()
        
        # Now you can use the key in your application
    print("Successfully loaded API Key.")
        # For security, we only show the first and last few characters
    print(f"Key starts with: {my_llm_key[:4]}... and ends with: ...{my_llm_key[-4:]}")
        
        # Example of using the key with a fictional API call
        # some_llm_library.authenticate(api_key=my_llm_key)
        
except ValueError as e:
    print(f"Error: {e}")
        
os.environ["GOOGLE_API_KEY"] = my_llm_key

Successfully loaded API Key.
Key starts with: AIza... and ends with: ...SZm4


In [14]:
from llm_factory import LoadGoogleLLM

llm=LoadGoogleLLM()

In [20]:
max_tokens = 100
temperature = 0
top_p = 0.05
echo = False
stop = ["\n\n"]

evaluation_dataset_AIgenerated=[]

import random
random_sample = random.sample(chunked_splits, 10)


for chunk in random_sample:
    chunkContent=chunk.page_content
    chunkId=chunk.metadata.get('doc_id')
    #user_prompt= f"Redacta una pregunta simple sobre el contenido del siguiente texto. #TEXTO:{chunkContent} #PREGUNTA:..."
    user_prompt = (
        "Genera UNA ÚNICA pregunta simple basada en el siguiente texto. "
        "NO incluyas introducciones, ni opciones, ni explicaciones. "
        "SOLO devuelve el texto de la pregunta y absolutamente nada más.\n\n"
        f"Texto: {chunkContent}"
    )
    messages = [HumanMessage(content=user_prompt)]
    #model_output = llm.invoke(messages)    
    #final_result = model_output.content.strip()
    
    #Invocamos al modelo
    model_output = llm.invoke(messages)      
    raw_content = model_output.content
    
    final_result = ""
    
    if isinstance(raw_content, list):
        for block in raw_content:
            # Si el bloque es texto directo
            if isinstance(block, str):
                final_result += block + " "
            # Si el bloque es un diccionario
            elif isinstance(block, dict):
                # Si detecta que es el pensamiento, se lo salta (continue)
                if block.get('type') == 'thinking' or 'thinking' in block:
                    continue
                # Si es el bloque de texto final, lo extrae
                elif 'text' in block:
                    final_result += block['text'] + " "
    else:
        # Si de casualidad viene como texto plano
        final_result = str(raw_content)

    # Limpiamos espacios o saltos de línea sobrantes
    final_result = final_result.strip()
        
    print(f"pedido: {chunkContent} \n --- \n respuesta: {final_result}")
    evaluation_dataset_AIgenerated.append({"text_chunk": f"{chunkContent}","question":f"{final_result}","ground_truth_doc_id":f"{chunkId}"})
        

pedido: En la **cuadratura de Gauss**, el problema se plantea en el **dominio unitario `[-1, 1]`**. Este método busca determinar tanto los coeficientes como los puntos de evaluación de la función. La idea es hallar los valores de abscisas `t1, t2, ...` y coeficientes `w1, w2, ...` para aproximar la integral `Integral from -1 to 1 of G(t) dt`. 
 --- 
 respuesta: ¿En qué dominio se plantea el problema de la cuadratura de Gauss?


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised InternalServerError: 500 Internal error encountered..
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 4.0 seconds as it raised InternalServerError: 500 Internal error encountered..
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 8.0 seconds as it raised InternalServerError: 500 Internal error encountered..


pedido: El objetivo principal es encontrar la **integral definida** `I` en `R`, dada por `I = Integral from X0 to Xn of f(x) dx`.
Basándose en la relación `Integral(f(x)dx) = Integral(Pn(x)dx) + Integral(En(x)dx)`, la integral definida `I` se evalúa como la suma:
`I = In + En`
Donde `In` se denomina **cuadratura** y `En` es el **error de truncamiento**.  
Todos los métodos de integración numérica comparten la estructura en la que la cuadratura `In` se expresa como una suma:
`In = sum(wj * f(xj))`
Los **coeficientes `wj`** se determinan según cada regla. El **orden de la regla de cuadratura** se define como el máximo grado del polinomio que dicha regla integra de forma exacta, es decir, para el cual `En = 0`.  
Existen dos tipos principales de cuadratura: 
 --- 
 respuesta: ¿Cómo se define el orden de la regla de cuadratura?


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised InternalServerError: 500 Internal error encountered..


pedido: La regla de dos puntos se puede generalizar a más puntos. Se proporciona una tabla con las abscisas (`ti`) y coeficientes (`wi`) para 2, 3 y 4 puntos de Gauss, y el orden de la derivada del error de truncamiento.  
*   **2 puntos:** Abscisas `+-0.577350269`, Coeficientes `1.0`, Orden de error `4`.
*   **3 puntos:** Abscisas `0`, `+-0.774596669`, Coeficientes `0.8888889`, `0.5555556`, Orden de error `6`.
*   **4 puntos:** Abscisas `+-0.339981044`, `+-0.861136312`, Coeficientes `0.6521452`, `0.3478548`, Orden de error `8`. 
 --- 
 respuesta: ¿Cuál es el orden de error de truncamiento para la regla de 2 puntos de Gauss?
pedido: Dada una matriz \(A\) (NxN), se asume que \(\lambda_i , v_i\) son los autovalores y autovectores de \(A\) (el índice \(i\) varía de 1 a N). Se verifica que \(A v_i = \lambda_i v_i \forall i = 1,...,N\). Si \(A\) es diagonalizable, los \( v_i\) son linealmente independientes y forman una base. Cualquier vector \(x\) puede escribirse como una combinación line

...

pedido: Partiendo de la definición `E1 = I - I1` y desarrollando la función primitiva `G(x)` en serie de Taylor alrededor de `xi`, se confirma que el **error en la regla de trapecios es O(h^3)**. 
 --- 
 respuesta: ¿Cuál es el error en la regla de trapecios?


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised InternalServerError: 500 Internal error encountered..


pedido: Para la regla de dos puntos, se proponen `w1*G(t1) + w2*G(t2)`. Los valores `w1, w2, t1, t2` se determinan de manera que el error de integración sea `R = 0` para polinomios de hasta 3er grado.
Las soluciones son: `t1 = -1/sqrt(3)`, `t2 = 1/sqrt(3)`, `w1 = 1`, `w2 = 1`.
La regla queda: `I = h * (f(c - h/sqrt(3)) + f(c + h/sqrt(3)))`. 
 --- 
 respuesta: ¿Para polinomios de hasta qué grado es el error de integración R = 0 en la regla de dos puntos?


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised InternalServerError: 500 Internal error encountered..
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 4.0 seconds as it raised InternalServerError: 500 Internal error encountered..


pedido: Un problema de **Valores y Vectores Propios** consiste en encontrar los vectores \(v\) tales que son direcciones invariantes de la transformación lineal dada por la matriz \(A\) (NxN). Esto se expresa como: \(A v = \lambda v \) siendo \(\lambda\) el escalar que cambia el módulo del vector cuya dirección permanece invariante. Se denomina **autovalor** \( \lambda \) y **autovector** \(v\). El sistema de ecuaciones puede escribirse de la forma: \((A - \lambda I) v = 0 \) donde interesan las soluciones \(v\) distintas de la trivial,\(v = 0\). Esto está garantizado sí y sólo sí \(det(A - \lambda I) = 0\). El determinante constituye un polinomio de grado N en el autovalor \(\lambda \), y se denomina **polinomio característico**. Las raíces de dicho polinomio son los autovalores \(\lambda \) de la matriz \(A\) para los cuales existen los autovectores o direcciones invariantes \(v\). Por cada valor propio existe al menos una dirección invariante dada por el autovector \(v\).  
Existen 

In [21]:
import pandas as pd

eval_df=pd.DataFrame(evaluation_dataset_AIgenerated) #Transformo lista en dataframe
eval_df.to_csv('../data/processed/evaluation_dataset_AIgenerated_revised.csv')

# Creando vector stores

In [6]:
GGUF_MODEL_PATH = "/home/nico/.cache/llama.cpp/nomic-embed-text-v2-moe.Q6_K.gguf"
PERSIST_DIRECTORY = "../db_chroma"
PERSIST_DIRECTORY_EMBEDDER = os.path.join(PERSIST_DIRECTORY,"nomic_v2")

embedding_encoder = LlamaCppEmbeddings(
    model_path=GGUF_MODEL_PATH,
    n_gpu_layers=0,   # Offload layers to GPU on your Windows machine. Set to 0 for CPU.
    n_batch=64,       # The number of documents to process in a single batch.
    n_ctx=512,        # The context size of the model.
    verbose=False      # Set to True to see more detailed llama.cpp output.
)
#Codificacion de los chunks y Creacion de la base de datos
if not os.path.isdir(PERSIST_DIRECTORY_EMBEDDER): #Si el directorio no existe
    os.mkdir(PERSIST_DIRECTORY_EMBEDDER)
    vectorStore = EmbeddDocsAndPersist(chunked_splits,embedding_encoder,PERSIST_DIRECTORY_EMBEDDER)
else:
    vectorStore = load_persisted_db(embedding_encoder,PERSIST_DIRECTORY_EMBEDDER)

Creando y persistiendo la base de datos de vectores...
Base de datos creada y guardada.


In [39]:
GGUF_MODEL_PATH = "/home/nico/.cache/llama.cpp/embeddinggemma-300m-Q4_0.gguf"
PERSIST_DIRECTORY = "../db_chroma"
PERSIST_DIRECTORY_EMBEDDER = os.path.join(PERSIST_DIRECTORY,"gemma")

embedding_encoder = LlamaCppEmbeddings(
    model_path=GGUF_MODEL_PATH,
    n_gpu_layers=0,   # Offload layers to GPU on your Windows machine. Set to 0 for CPU.
    n_batch=64,       # The number of documents to process in a single batch.
    n_ctx=512,        # The context size of the model.
    verbose=False      # Set to True to see more detailed llama.cpp output.
)
#Codificacion de los chunks y Creacion de la base de datos
if not os.path.isdir(PERSIST_DIRECTORY_EMBEDDER): #Si el directorio no existe
    os.mkdir(PERSIST_DIRECTORY_EMBEDDER)
    vectorStore = EmbeddDocsAndPersist(chunked_splits,embedding_encoder,PERSIST_DIRECTORY_EMBEDDER)
else:
    vectorStore = load_persisted_db(embedding_encoder,PERSIST_DIRECTORY_EMBEDDER)

Creando y persistiendo la base de datos de vectores...
Base de datos creada y guardada.


In [40]:
GGUF_MODEL_PATH = "/home/nico/.cache/llama.cpp/bge-m3-q8_0.gguf"
PERSIST_DIRECTORY = "../db_chroma"
PERSIST_DIRECTORY_EMBEDDER = os.path.join(PERSIST_DIRECTORY,"bge")

embedding_encoder = LlamaCppEmbeddings(
    model_path=GGUF_MODEL_PATH,
    n_gpu_layers=0,   # Offload layers to GPU on your Windows machine. Set to 0 for CPU.
    n_batch=64,       # The number of documents to process in a single batch.
    n_ctx=512,        # The context size of the model.
    verbose=False      # Set to True to see more detailed llama.cpp output.
)
#Codificacion de los chunks y Creacion de la base de datos
if not os.path.isdir(PERSIST_DIRECTORY_EMBEDDER): #Si el directorio no existe
    os.mkdir(PERSIST_DIRECTORY_EMBEDDER)
    vectorStore = EmbeddDocsAndPersist(chunked_splits,embedding_encoder,PERSIST_DIRECTORY_EMBEDDER)
else:
    vectorStore = load_persisted_db(embedding_encoder,PERSIST_DIRECTORY_EMBEDDER)

Creando y persistiendo la base de datos de vectores...
Base de datos creada y guardada.


In [41]:
GGUF_MODEL_PATH = "/home/nico/.cache/llama.cpp/all-MiniLM-L6-v2-Q8_0.gguf"
PERSIST_DIRECTORY = "../db_chroma"
PERSIST_DIRECTORY_EMBEDDER = os.path.join(PERSIST_DIRECTORY,"allMiniLM")

embedding_encoder = LlamaCppEmbeddings(
    model_path=GGUF_MODEL_PATH,
    n_gpu_layers=0,   # Offload layers to GPU on your Windows machine. Set to 0 for CPU.
    n_batch=64,       # The number of documents to process in a single batch.
    n_ctx=512,        # The context size of the model.
    verbose=False      # Set to True to see more detailed llama.cpp output.
)
#Codificacion de los chunks y Creacion de la base de datos
if not os.path.isdir(PERSIST_DIRECTORY_EMBEDDER): #Si el directorio no existe
    os.mkdir(PERSIST_DIRECTORY_EMBEDDER)
    vectorStore = EmbeddDocsAndPersist(chunked_splits,embedding_encoder,PERSIST_DIRECTORY_EMBEDDER)
else:
    vectorStore = load_persisted_db(embedding_encoder,PERSIST_DIRECTORY_EMBEDDER)

Creando y persistiendo la base de datos de vectores...
Base de datos creada y guardada.


In [42]:
GGUF_MODEL_PATH = "/home/nico/.cache/llama.cpp/mxbai-embed-large-v1.Q8_0.gguf"
PERSIST_DIRECTORY = "../db_chroma"
PERSIST_DIRECTORY_EMBEDDER = os.path.join(PERSIST_DIRECTORY,"mxbai")

embedding_encoder = LlamaCppEmbeddings(
    model_path=GGUF_MODEL_PATH,
    n_gpu_layers=0,   # Offload layers to GPU on your Windows machine. Set to 0 for CPU.
    n_batch=64,       # The number of documents to process in a single batch.
    n_ctx=512,        # The context size of the model.
    verbose=False      # Set to True to see more detailed llama.cpp output.
)
#Codificacion de los chunks y Creacion de la base de datos
if not os.path.isdir(PERSIST_DIRECTORY_EMBEDDER): #Si el directorio no existe
    os.mkdir(PERSIST_DIRECTORY_EMBEDDER)
    vectorStore = EmbeddDocsAndPersist(chunked_splits,embedding_encoder,PERSIST_DIRECTORY_EMBEDDER)
else:
    vectorStore = load_persisted_db(embedding_encoder,PERSIST_DIRECTORY_EMBEDDER)

Creando y persistiendo la base de datos de vectores...
Base de datos creada y guardada.


In [43]:
GGUF_MODEL_PATH = "/home/nico/.cache/llama.cpp/Qwen3-Embedding-0.6B-Q8_0.gguf"
PERSIST_DIRECTORY = "../db_chroma"
PERSIST_DIRECTORY_EMBEDDER = os.path.join(PERSIST_DIRECTORY,"qwen3")

embedding_encoder = LlamaCppEmbeddings(
    model_path=GGUF_MODEL_PATH,
    n_gpu_layers=0,   # Offload layers to GPU on your Windows machine. Set to 0 for CPU.
    n_batch=64,       # The number of documents to process in a single batch.
    n_ctx=512,        # The context size of the model.
    verbose=False      # Set to True to see more detailed llama.cpp output.
)
#Codificacion de los chunks y Creacion de la base de datos
if not os.path.isdir(PERSIST_DIRECTORY_EMBEDDER): #Si el directorio no existe
    os.mkdir(PERSIST_DIRECTORY_EMBEDDER)
    vectorStore = EmbeddDocsAndPersist(chunked_splits,embedding_encoder,PERSIST_DIRECTORY_EMBEDDER)
else:
    vectorStore = load_persisted_db(embedding_encoder,PERSIST_DIRECTORY_EMBEDDER)

Creando y persistiendo la base de datos de vectores...
Base de datos creada y guardada.


## Pruebas

In [7]:
homepath='/home/nico/.cache/llama.cpp/'
vectorStores=[{'name':'nomic_v2','path':'../db_chroma/nomic_v2','GGUF_MODEL_PATH':homepath+'nomic-embed-text-v2-moe.Q6_K.gguf'},
              {'name':'gemma','path':'../db_chroma/gemma','GGUF_MODEL_PATH':homepath+'embeddinggemma-300m-Q4_0.gguf'},
              {'name':'bge','path':'../db_chroma/bge','GGUF_MODEL_PATH':homepath+'bge-m3-q8_0.gguf'},
              {'name':'allMiniLM','path':'../db_chroma/allMiniLM','GGUF_MODEL_PATH':homepath+'all-MiniLM-L6-v2-Q8_0.gguf'},
              {'name':'mxbai','path':'../db_chroma/mxbai','GGUF_MODEL_PATH':homepath+'mxbai-embed-large-v1.Q8_0.gguf'},
              {'name':'qwen3','path':'../db_chroma/qwen3','GGUF_MODEL_PATH':homepath+'Qwen3-Embedding-0.6B-Q8_0.gguf'}
             ]

In [8]:
import pandas as pd

eval_df=pd.read_csv('../data/processed/evaluation_dataset_AIgenerated_revised.csv')
eval_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Unnamed: 0           10 non-null     int64 
 1   text_chunk           10 non-null     object
 2   question             10 non-null     object
 3   ground_truth_doc_id  10 non-null     object
dtypes: int64(1), object(3)
memory usage: 452.0+ bytes


In [9]:
from utils import evaluate_vectorstore_as_retriever


metrics=[]

for vs in vectorStores:
    modelName=vs.get('name')
    print(f"Creando modelo encoder {modelName}...")
    embedding_encoder = LlamaCppEmbeddings(
    model_path=vs.get('GGUF_MODEL_PATH'),
    n_gpu_layers=0,   # Offload layers to GPU on your Windows machine. Set to 0 for CPU.
    n_batch=64,       # The number of documents to process in a single batch.
    n_ctx=512,        # The context size of the model.
    verbose=False      # Set to True to see more detailed llama.cpp output.
    )
    print(f"Cargando vector store {vs.get('path')}...")
    vectorStore = load_persisted_db(embedding_encoder,vs.get('path'))
    total_records = len(vectorStore.get()['documents'])

    for k in [3,5,10,15]:
        results=evaluate_vectorstore_as_retriever(eval_df[['question','ground_truth_doc_id']].to_dict(orient='records'),
                                      vectorStore, 
                                      k=k)
        metrics.append({
                        'model':modelName,
                        'k':k,
                        'k % chunks':np.round(k*100/total_records),
                        'hit rate':results.get('hit_rate'),
                        'mrr':results.get('mrr')
                        }
        )

pd.DataFrame(metrics)

Creando modelo encoder nomic_v2...
Cargando vector store ./db_chroma/nomic_v2...
Cargando base de datos persistente...
Base de datos cargada.


/home/nico/Documentos/Trabajo/Fing/EDUBOT/RAG-Bot/LLM-RAG/src/utils.py:53: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorStore = Chroma(


Starting evaluation for k=3...
Starting evaluation for k=5...
Starting evaluation for k=10...
Starting evaluation for k=15...
Creando modelo encoder gemma...
Cargando vector store ./db_chroma/gemma...
Cargando base de datos persistente...
Base de datos cargada.
Starting evaluation for k=3...
Starting evaluation for k=5...
Starting evaluation for k=10...
Starting evaluation for k=15...
Creando modelo encoder bge...
Cargando vector store ./db_chroma/bge...
Cargando base de datos persistente...
Base de datos cargada.
Starting evaluation for k=3...
Starting evaluation for k=5...
Starting evaluation for k=10...
Starting evaluation for k=15...
Creando modelo encoder allMiniLM...
Cargando vector store ./db_chroma/allMiniLM...
Cargando base de datos persistente...
Base de datos cargada.
Starting evaluation for k=3...
Starting evaluation for k=5...
Starting evaluation for k=10...
Starting evaluation for k=15...
Creando modelo encoder mxbai...
Cargando vector store ./db_chroma/mxbai...
Cargando 

,model,k,k % chunks,hit rate,mrr
0,nomic_v2,3,11.0,70.00%,0.6000
1,nomic_v2,5,18.0,80.00%,0.6250
2,nomic_v2,10,36.0,80.00%,0.6250
3,nomic_v2,15,54.0,80.00%,0.6250
4,gemma,3,11.0,70.00%,0.6000
5,gemma,5,18.0,80.00%,0.6250
6,gemma,10,36.0,80.00%,0.6250
7,gemma,15,54.0,90.00%,0.6341
8,bge,3,11.0,60.00%,0.5500
9,bge,5,18.0,60.00%,0.5500
